In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


model = mlflow.sklearn.load_model(
    "models:/lead_scoring_model@champion"
)


df = spark.read.table("medallion.gold.lead_scoring_dataset")

pdf = df.toPandas()


X = pdf.drop(
    columns=[
        "convertido",
        "lead_id",
        "processing_timestamp"
    ],
    errors="ignore"
)


scores = model.predict_proba(X)[:, 1]

pdf["lead_score"] = scores

df_scores = pdf[
    [
        "lead_id",
        "data_hora",
        "lead_score"
    ]
]


spark.createDataFrame(df_scores) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion.gold.lead_scores")


print("Lead scoring atualizado com sucesso.")

Lead scoring atualizado com sucesso.
